# Imports

In [ ]:
%pip install pandas seaborn matplotlib imbalanced-learn scikit-learn scikit-optimize xgboost lightgbm catboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import seaborn as sns

import matplotlib.pyplot as plt
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import time

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance
import pickle

# Exploratory Data Analysis

## Dataset reading

In [ ]:
file_path = r"C:\Users\Jorge Ferreira\Desktop\Credit Risk Prediction\Loan_default.csv"
df = pd.read_csv(file_path)

In [ ]:
selected_features = [
    "Age", "Income", "LoanAmount", "CreditScore",
    "MonthsEmployed", "NumCreditLines", "InterestRate"
]
target = "Default"

In [ ]:
print("Dataset loaded successfully. First ten rows:")
display(df.head(10))

print(f"Dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns.")
print("Dataset structure, data types, and non-null values:")
df.info()

In [ ]:
print("Descriptive statistics for the selected features:")
display(df[selected_features].describe().T)

null_report = pd.DataFrame({
    "Null Count": df[selected_features + [target]].isnull().sum(),
    "Null Percentage": (df[selected_features + [target]].isnull().mean() * 100).round(2)
})
print("Null-value counts and percentages for selected features and target:")
display(null_report)

exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates:,}")

profile_columns = selected_features + [target]
duplicate_profiles = df.duplicated(subset=profile_columns, keep=False).sum()
print(
    f"Rows belonging to duplicated profiles based on the seven selected features "
    f"and {target} (excluding LoanID): {duplicate_profiles:,}"
)

In [ ]:
loan_id_unique = df["LoanID"].is_unique
print(f"LoanID is unique: {loan_id_unique}")

default_distribution = pd.DataFrame({
    "Count": df[target].value_counts(),
    "Proportion (%)": (df[target].value_counts(normalize=True) * 100).round(2)
})
print("Default class counts and proportions:")
display(default_distribution)

In [ ]:
correlation_matrix = df[selected_features + [target]].corr(numeric_only=True)
print("Correlation matrix for selected numerical features and Default:")
display(correlation_matrix)

correlation_pairs = (
    correlation_matrix.where(
        __import__("numpy").triu(__import__("numpy").ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index(name="Correlation")
    .rename(columns={"level_0": "Feature 1", "level_1": "Feature 2"})
    .sort_values("Correlation", ascending=False)
)

print("Feature correlation pairs sorted in descending order:")
display(correlation_pairs)

# Feature Engineering

In [ ]:
X = df[selected_features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
target_distributions = pd.DataFrame({
    "Full dataset": y.value_counts(normalize=True).mul(100).round(2),
    "Training set": y_train.value_counts(normalize=True).mul(100).round(2),
    "Test set": y_test.value_counts(normalize=True).mul(100).round(2)
})

target_counts = pd.DataFrame({
    "Full dataset": y.value_counts(),
    "Training set": y_train.value_counts(),
    "Test set": y_test.value_counts()
})

display(target_counts)
display(target_distributions)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x=y_train, ax=axes[0], hue=y_train, palette="Blues", legend=False)
axes[0].set_title("Training Set Target Distribution")
axes[0].set_xlabel("Default")
axes[0].set_ylabel("Count")

sns.countplot(x=y_test, ax=axes[1], hue=y_test, palette="Oranges", legend=False)
axes[1].set_title("Test Set Target Distribution")
axes[1].set_xlabel("Default")
axes[1].set_ylabel("Count")

for ax in axes:
    for container in ax.containers:
        ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()

# Model Development

In [ ]:
print("=" * 90)
print("CROSS-VALIDATED MODEL DEVELOPMENT")
print("=" * 90)
print("Pipeline order: StandardScaler → SMOTE(sampling_strategy=0.5) → Model")
print("GridSearchCV receives the original X_train and y_train only.")
print("Scoring metric: Average Precision (PR-AUC)")
print("Cross-validation: StratifiedKFold(n_splits=3, shuffle=True, random_state=42)")
print("=" * 90)

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

models_and_grids = [
    (
        "Logistic Regression",
        LogisticRegression(max_iter=5000, random_state=42),
        [
            {
                "model__solver": ["liblinear"],
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
                "model__class_weight": [None, "balanced"]
            },
            {
                "model__solver": ["lbfgs"],
                "model__penalty": ["l2"],
                "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
                "model__class_weight": [None, "balanced"]
            },
            {
                "model__solver": ["saga"],
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"]
            }
        ]
    ),
    (
        "Linear SVC",
        LinearSVC(max_iter=8000, random_state=42),
        [
            {
                "model__penalty": ["l2"],
                "model__loss": ["hinge", "squared_hinge"],
                "model__dual": [True],
                "model__C": [0.001, 0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"],
                "model__tol": [1e-4, 1e-3]
            },
            {
                "model__penalty": ["l1"],
                "model__loss": ["squared_hinge"],
                "model__dual": [False],
                "model__C": [0.001, 0.01, 0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"],
                "model__tol": [1e-4, 1e-3]
            }
        ]
    ),
    (
        "Decision Tree",
        DecisionTreeClassifier(random_state=42),
        {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__splitter": ["best", "random"],
            "model__max_depth": [3, 6, 10, None],
            "model__min_samples_split": [2, 5, 10, 20],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__max_features": [None, "sqrt", "log2"],
            "model__class_weight": [None, "balanced"],
            "model__ccp_alpha": [0.0, 0.001, 0.01]
        }
    ),
    (
        "Random Forest",
        RandomForestClassifier(random_state=42, n_jobs=1),
        {
            "model__n_estimators": [200, 400],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 8, 12, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 5],
            "model__max_features": ["sqrt", "log2", 0.5],
            "model__bootstrap": [True, False],
            "model__class_weight": [None, "balanced", "balanced_subsample"]
        }
    ),
    (
        "LightGBM",
        LGBMClassifier(random_state=42, n_jobs=1, verbosity=-1),
        {
            "model__n_estimators": [200, 400],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 63],
            "model__max_depth": [-1, 6, 10],
            "model__min_child_samples": [10, 20, 40],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
            "model__reg_alpha": [0.0, 0.1],
            "model__reg_lambda": [0.0, 1.0, 5.0]
        }
    ),
    (
        "XGBoost",
        XGBClassifier(
            random_state=42,
            n_jobs=1,
            eval_metric="logloss",
            tree_method="hist"
        ),
        {
            "model__n_estimators": [200, 400],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_child_weight": [1, 3, 5],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
            "model__gamma": [0, 0.1, 0.3],
            "model__reg_lambda": [1.0, 3.0, 10.0]
        }
    ),
    (
        "CatBoost",
        CatBoostClassifier(
            random_state=42,
            thread_count=1,
            verbose=0,
            allow_writing_files=False
        ),
        {
            "model__iterations": [200, 400],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__depth": [4, 6, 8],
            "model__l2_leaf_reg": [3, 5, 7, 10],
            "model__border_count": [64, 128, 254],
            "model__random_strength": [1.0, 2.0],
            "model__bagging_temperature": [0, 1]
        }
    ),
    (
        "AdaBoost",
        AdaBoostClassifier(random_state=42),
        {
            "model__n_estimators": [100, 200, 400],
            "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
            "model__algorithm": ["SAMME"]
        }
    ),
    (
        "Random Forest (Balanced Focus)",
        RandomForestClassifier(random_state=42, n_jobs=1, class_weight="balanced_subsample"),
        {
            "model__n_estimators": [300, 500],
            "model__max_depth": [6, 10, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 3, 5],
            "model__max_features": ["sqrt", 0.5],
            "model__bootstrap": [True]
        }
    )
]

results = []
best_models = {}

In [ ]:
for model_name, model, param_grid in models_and_grids:
    print(f"\n{'─' * 90}")
    print(f"Training: {model_name}")
    print(f"Hyperparameter combinations: {len(list(ParameterGrid(param_grid)))}")
    print("Running 3-fold stratified GridSearchCV...")

    pipeline = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(sampling_strategy=0.5, random_state=42)),
        ("model", model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )

    start_time = time.perf_counter()
    grid_search.fit(X_train, y_train)
    training_time = time.perf_counter() - start_time

    best_model = grid_search.best_estimator_
    best_models[model_name] = best_model

    y_pred = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    test_pr_auc = average_precision_score(y_test, y_score)
    test_roc_auc = roc_auc_score(y_test, y_score)

    best_cv_std = grid_search.cv_results_["std_test_score"][grid_search.best_index_]

    print(f"Best hyperparameters: {grid_search.best_params_}")
    print(f"Best CV PR-AUC:      {grid_search.best_score_:.4f} ± {best_cv_std:.4f}")
    print(f"Training time:       {training_time:.2f} seconds")
    print(f"Test Precision:      {precision:.4f}")
    print(f"Test Recall:         {recall:.4f}")
    print(f"Test F1-Score:       {f1:.4f}")
    print(f"Test PR-AUC:         {test_pr_auc:.4f}")
    print(f"Test ROC-AUC:        {test_roc_auc:.4f}")

    results.append({
        "Model": model_name,
        "Best Hyperparameters": grid_search.best_params_,
        "CV PR-AUC": grid_search.best_score_,
        "CV PR-AUC Std": best_cv_std,
        "Training Time (sec)": training_time,
        "Test Precision": precision,
        "Test Recall": recall,
        "Test F1-Score": f1,
        "Test PR-AUC": test_pr_auc,
        "Test ROC-AUC": test_roc_auc
    })

In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values("Test PR-AUC", ascending=False)
    .reset_index(drop=True)
)

print(f"\n{'=' * 90}")
print("FINAL MODEL COMPARISON — SORTED BY TEST PR-AUC")
print(f"{'=' * 90}")

print(
    results_df.to_string(
        index=False,
        formatters={
            "CV PR-AUC": "{:.4f}".format,
            "CV PR-AUC Std": "{:.4f}".format,
            "Training Time (sec)": "{:.2f}".format,
            "Test Precision": "{:.4f}".format,
            "Test Recall": "{:.4f}".format,
            "Test F1-Score": "{:.4f}".format,
            "Test PR-AUC": "{:.4f}".format,
            "Test ROC-AUC": "{:.4f}".format
        }
    )
)

print(f"\nBest model by Test PR-AUC: {results_df.loc[0, 'Model']}")
print(f"Best Test PR-AUC: {results_df.loc[0, 'Test PR-AUC']:.4f}")

In [ ]:
print(f"\n{'=' * 90}")
print("FINAL MODEL COMPARISON — SORTED BY TEST PR-AUC")
print(f"{'=' * 90}")

print(
    results_df.to_string(
        index=False,
        formatters={
            "CV PR-AUC": "{:.4f}".format,
            "CV PR-AUC Std": "{:.4f}".format,
            "Training Time (sec)": "{:.2f}".format,
            "Test Precision": "{:.4f}".format,
            "Test Recall": "{:.4f}".format,
            "Test F1-Score": "{:.4f}".format,
            "Test PR-AUC": "{:.4f}".format,
            "Test ROC-AUC": "{:.4f}".format
        }
    )
)

print(f"\nBest model by Test PR-AUC: {results_df.loc[0, 'Model']}")
print(f"Best Test PR-AUC: {results_df.loc[0, 'Test PR-AUC']:.4f}")

In [ ]:
k_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
test_default_rate = y_test.mean()

evaluation_results = []
ranking_curves = []
permutation_results = []

for model_name, fitted_pipeline in best_models.items():
    if hasattr(fitted_pipeline, "predict_proba"):
        scores = fitted_pipeline.predict_proba(X_test)[:, 1]
    else:
        scores = fitted_pipeline.decision_function(X_test)

    predictions = fitted_pipeline.predict(X_test)

    model_metrics = {
        "Model": model_name,
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, scores),
        "PR-AUC": average_precision_score(y_test, scores),
        "Predicted Positive Rate": predictions.mean()
    }

    print(f"\n{'=' * 70}\n{model_name}")
    print(f"Precision: {model_metrics['Precision']:.4f}")
    print(f"Recall:    {model_metrics['Recall']:.4f}")
    print(f"F1-Score:  {model_metrics['F1-Score']:.4f}")
    print(f"ROC-AUC:   {model_metrics['ROC-AUC']:.4f}")
    print(f"PR-AUC:    {model_metrics['PR-AUC']:.4f}")

    ranked_indices = np.argsort(-scores)

    for k in k_values:
        top_n = int(np.ceil(len(y_test) * k))
        top_k_actuals = y_test.iloc[ranked_indices[:top_n]]

        precision_at_k = top_k_actuals.mean()
        recall_at_k = top_k_actuals.sum() / y_test.sum()
        lift_at_k = precision_at_k / test_default_rate

        label = f"{int(k * 100)}%"
        model_metrics[f"Precision@{label}"] = precision_at_k
        model_metrics[f"Recall@{label}"] = recall_at_k
        model_metrics[f"Lift@{label}"] = lift_at_k

        ranking_curves.append({
            "Model": model_name,
            "Top Population (%)": int(k * 100),
            "Precision@k": precision_at_k,
            "Recall@k": recall_at_k,
            "Lift@k": lift_at_k
        })

    evaluation_results.append(model_metrics)

    print("Calculating permutation importance...")
    permutation = permutation_importance(
        fitted_pipeline,
        X_test,
        y_test,
        scoring="average_precision",
        n_repeats=5,
        random_state=42,
        n_jobs=-1
    )

    permutation_results.append(
        pd.DataFrame({
            "Model": model_name,
            "Feature": selected_features,
            "Importance Mean": permutation.importances_mean,
            "Importance Std": permutation.importances_std
        })
    )

In [ ]:
evaluation_df = (
    pd.DataFrame(evaluation_results)
    .sort_values("PR-AUC", ascending=False)
    .reset_index(drop=True)
)

model_selection_df = (
    results_df[
        ["Model", "Best Hyperparameters", "CV PR-AUC", "CV PR-AUC Std", "Training Time (sec)", "Test PR-AUC", "Test ROC-AUC"]
    ].copy()
)

model_selection_df["CV-to-Test PR-AUC Difference"] = (
    model_selection_df["CV PR-AUC"] - model_selection_df["Test PR-AUC"]
)

permutation_importance_df = pd.concat(permutation_results, ignore_index=True)
ranking_curves_df = pd.DataFrame(ranking_curves)

print("\nMASTER EVALUATION RESULTS")
display(evaluation_df)

print("\nBEST PARAMETERS AND CV RESULTS")
display(model_selection_df.sort_values("Test PR-AUC", ascending=False))

# Grouped bar chart of core metrics
core_metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC"]
plot_metrics = evaluation_df.set_index("Model")[core_metrics]


In [ ]:

ax = plot_metrics.plot(
    kind="bar",
    figsize=(16, 7),
    width=0.85,
    colormap="tab10"
)
ax.set_title("Model Comparison: Core Test Metrics")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.set_ylim(0, 1)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Ranking performance curves
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for model_name, group in ranking_curves_df.groupby("Model"):
    group = group.sort_values("Top Population (%)")
    axes[0].plot(group["Top Population (%)"], group["Precision@k"], marker="o", label=model_name)
    axes[1].plot(group["Top Population (%)"], group["Recall@k"], marker="o", label=model_name)
    axes[2].plot(group["Top Population (%)"], group["Lift@k"], marker="o", label=model_name)

axes[0].set_title("Precision@k")
axes[1].set_title("Recall@k")
axes[2].set_title("Lift@k")

for axis in axes:
    axis.set_xlabel("Top Population (%)")
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)

axes[0].set_ylabel("Precision")
axes[1].set_ylabel("Recall")
axes[2].set_ylabel("Lift")

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

for axis, (model_name, fitted_pipeline) in zip(axes, best_models.items()):
    model_predictions = fitted_pipeline.predict(X_test)
    matrix = confusion_matrix(y_test, model_predictions)

    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=["No Default", "Default"]
    ).plot(ax=axis, colorbar=False)

    axis.set_title(model_name)

plt.suptitle("Confusion Matrices", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Signed standardized coefficients for linear models
linear_models = ["Logistic Regression", "Linear SVC"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for axis, model_name in zip(axes, linear_models):
    fitted_model = best_models[model_name].named_steps["model"]
    coefficients = pd.Series(
        fitted_model.coef_.ravel(),
        index=selected_features
    ).sort_values()

    colors = np.where(coefficients >= 0, "steelblue", "firebrick")
    coefficients.plot(kind="barh", ax=axis, color=colors)

    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_title(f"{model_name}: Signed Standardized Coefficients")
    axis.set_xlabel("Coefficient")

plt.tight_layout()
plt.show()

In [ ]:
# Native feature importance for tree and boosting models
tree_boost_models = [
    "Decision Tree", "Random Forest", "LightGBM",
    "XGBoost", "CatBoost", "AdaBoost"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for axis, model_name in zip(axes, tree_boost_models):
    fitted_model = best_models[model_name].named_steps["model"]

    importance = pd.Series(
        fitted_model.feature_importances_,
        index=selected_features
    ).sort_values()

    importance.plot(kind="barh", ax=axis, color="teal")
    axis.set_title(f"{model_name}: Native Feature Importance")
    axis.set_xlabel("Importance")

plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance for every model
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for axis, model_name in zip(axes, best_models):
    importance_data = (
        permutation_importance_df[
            permutation_importance_df["Model"] == model_name
        ]
        .sort_values("Importance Mean")
    )

    axis.barh(
        importance_data["Feature"],
        importance_data["Importance Mean"],
        xerr=importance_data["Importance Std"],
        color="mediumpurple",
        alpha=0.85
    )
    axis.set_title(f"{model_name}: Permutation Importance")
    axis.set_xlabel("Decrease in PR-AUC")

plt.tight_layout()
plt.show()

final_comparison_df = (
    evaluation_df
    .merge(
        model_selection_df[
            ["Model", "CV PR-AUC", "CV-to-Test PR-AUC Difference"]
        ],
        on="Model",
        how="left"
    )
    .sort_values(["PR-AUC", "ROC-AUC"], ascending=False)
    .reset_index(drop=True)
)

print("\nFINAL RANKING — PRIMARY SORT: TEST PR-AUC")
display(
    final_comparison_df[
        ["Model", "PR-AUC", "ROC-AUC", "Precision", "Recall",
         "F1-Score", "Predicted Positive Rate",
         "CV PR-AUC", "CV-to-Test PR-AUC Difference"]
    ]
)

print(
    "\nOperational interpretation: Precision@k measures the default rate among "
    "the highest-risk applicants selected for action, while Recall@k measures "
    "the share of all defaults captured. Increasing k generally captures more "
    "defaults (higher recall) but lowers precision and lift. Model selection "
    "should prioritize PR-AUC and business capacity rather than ROC-AUC alone."
)

In [ ]:
current_timestamp = pd.Timestamp.now()
print("Current timestamp:", current_timestamp)

In [ ]:
best_metrics = results_df.loc[results_df["Test PR-AUC"].idxmax()]
best_model_name = best_metrics["Model"]
best_model = best_models[best_model_name]
model_file = f"best_credit_risk_model_{current_timestamp}.pkl"

with open(model_file, "wb") as file:
    pickle.dump(best_model, file)

print(f"Saved best model to: {model_file}")
print(f"Model selected by Test PR-AUC: {best_model_name}")
print(f"Best hyperparameters: {best_metrics['Best Hyperparameters']}")
print(f"CV PR-AUC: {best_metrics['CV PR-AUC']:.4f}")
print(f"Test PR-AUC: {best_metrics['Test PR-AUC']:.4f}")
print(f"Test ROC-AUC: {best_metrics['Test ROC-AUC']:.4f}")
print(f"Test precision: {best_metrics['Test Precision']:.4f}")
print(f"Test recall: {best_metrics['Test Recall']:.4f}")
print(f"Test F1-score: {best_metrics['Test F1-Score']:.4f}")